# Lab 18.3: Edit a Memory, Count the Ripples

Chapter 18, Steering, Editing and Monitoring Models. Run-only notebook in plain numpy (about 10 seconds). A transformer's feed-forward layer can be read as a key-value memory: a subject's representation (the key) is mapped by a weight matrix to a stored value. ROME (Meng et al., 2022) edits one fact with a rank-one change to that matrix. Here the memory is a toy: 100 subjects, 20 possible facts, keys drawn from a correlated distribution, and a matrix W that stores every subject's fact exactly. You will edit one fact two ways and check the four things Chapter 18 asks of any edit: efficacy, generalization, specificity and ripple consistency, then make many edits in sequence.

In [1]:
import numpy as np
rng = np.random.default_rng(7)
d, n, labels = 128, 100, 20
A = rng.standard_normal((d, d)) / np.sqrt(d)
scales = np.linspace(2.5, 0.3, d)
def draw_keys(m):
    return (rng.standard_normal((m, d)) * scales) @ A.T
K = draw_keys(n)                                   # stored subjects
BG = draw_keys(20000)                              # background keys to estimate the key covariance
CODE = rng.standard_normal((labels, d))            # one value vector per fact
fact = rng.integers(0, labels, n)                  # which fact each subject holds
V = CODE[fact]
W = V.T @ np.linalg.pinv(K.T)                      # exact storage: W @ K[i] == V[i]
read = lambda W, keys: np.argmax((keys @ W.T) @ CODE.T, axis=1)
print("stored facts read back correctly:", np.mean(read(W, K) == fact))
C = BG.T @ BG / len(BG)
Ci = np.linalg.inv(C)

stored facts read back correctly: 1.0


In [2]:
def edit(W, k, new_value, metric):
    q = metric @ k
    return W + np.outer(new_value - W @ k, q) / (q @ k)

def report(W2, i, new_label, name):
    others = np.arange(n) != i
    para = K[i] + 0.15 * np.linalg.norm(K[i]) / np.sqrt(d) * rng.standard_normal((50, d))
    eff = read(W2, K[i:i + 1])[0] == new_label
    gen = np.mean(read(W2, para) == new_label)
    spec = np.mean(read(W2, K[others]) == fact[others])
    drift = np.mean(np.linalg.norm(K[others] @ W2.T - V[others], axis=1) / np.linalg.norm(V[others], axis=1))
    print(f"{name:28s} efficacy {eff!s:5s}  paraphrases {gen:.2f}  others unchanged {spec:.2f}  mean drift {drift:.3f}")

## Step 1: one edit, two metrics

**Checkpoint 1:** both edits store the new fact exactly (efficacy True) and every other subject still reads back its old fact. In this toy the plain update generalizes better to noisy paraphrases of the subject (about 0.9 against 0.6), while the covariance-weighted (ROME-style) update disturbs the other subjects' stored values less (mean drift about 0.14 against 0.20). Note that the read-out hides the drift: a fact can be "unchanged" at the top of the list while its stored value has moved by a fifth. Neither number alone describes the edit.

In [3]:
i, new_label = 0, (fact[0] + 1) % labels
for name, M in (("plain rank-one", np.eye(d)), ("covariance-weighted (ROME)", Ci)):
    report(edit(W, K[i], CODE[new_label], M), i, new_label, name)

plain rank-one               efficacy True   paraphrases 0.90  others unchanged 1.00  mean drift 0.198
covariance-weighted (ROME)   efficacy True   paraphrases 0.58  others unchanged 1.00  mean drift 0.137


## Step 2: a ripple the edit does not make

Suppose subject 1 is defined by subject 0 (as "the parent company of" a bank is defined by the bank): whenever subject 0's fact changes, subject 1's should change to match. A local edit knows nothing of that rule. **Checkpoint 2:** after the edit, subject 0 reads the new fact and subject 1 still reads its old one; the knowledge is now inconsistent. This is the failure RippleEdits measures (Cohen et al., 2024).

In [4]:
fact_dep = fact.copy(); fact_dep[1] = fact[0]       # make subject 1 depend on subject 0
V2 = CODE[fact_dep]; Wd = V2.T @ np.linalg.pinv(K.T)
W_e = edit(Wd, K[0], CODE[new_label], Ci)
print("subject 0 now reads new fact:", read(W_e, K[0:1])[0] == new_label)
print("subject 1 (should follow) reads new fact:", read(W_e, K[1:2])[0] == new_label,
      "| still reads old fact:", read(W_e, K[1:2])[0] == fact[0])

subject 0 now reads new fact: True
subject 1 (should follow) reads new fact: False | still reads old fact: True


## Step 3: many edits in sequence

Apply edits one after another, each to a different subject, and track how many earlier edits still hold and how many untouched subjects still read their original fact. **Checkpoint 3:** both updates hold the first ten edits. With the plain update, fewer than half of the earlier edits still hold after 80 edits and untouched facts start to break; the covariance-weighted update degrades more slowly (about 0.85 of earlier edits holding after 80). The pattern, slow erosion as edits accumulate, is a toy echo of the forgetting reported for long sequences of ROME and MEMIT edits in real models (Gupta et al., 2024).

In [5]:
for name, M in (("plain", np.eye(d)), ("covariance-weighted", Ci)):
    Ws, targets = W.copy(), {}
    print(name)
    for step in range(1, 81):
        j = step - 1
        lab = (fact[j] + 3) % labels
        Ws = edit(Ws, K[j], CODE[lab], M); targets[j] = lab
        if step in (1, 10, 20, 40, 60, 80):
            kept = np.mean([read(Ws, K[t:t + 1])[0] == l for t, l in targets.items()])
            untouched = [t for t in range(n) if t not in targets]
            intact = np.mean(read(Ws, K[untouched]) == fact[untouched])
            print(f"  after {step:3d} edits: earlier edits holding {kept:.2f}, untouched facts intact {intact:.2f}")

plain
  after   1 edits: earlier edits holding 1.00, untouched facts intact 1.00
  after  10 edits: earlier edits holding 1.00, untouched facts intact 0.99
  after  20 edits: earlier edits holding 0.95, untouched facts intact 0.96
  after  40 edits: earlier edits holding 0.65, untouched facts intact 1.00
  after  60 edits: earlier edits holding 0.57, untouched facts intact 0.85
  after  80 edits: earlier edits holding 0.45, untouched facts intact 0.85
covariance-weighted
  after   1 edits: earlier edits holding 1.00, untouched facts intact 1.00
  after  10 edits: earlier edits holding 1.00, untouched facts intact 1.00
  after  20 edits: earlier edits holding 1.00, untouched facts intact 1.00
  after  40 edits: earlier edits holding 0.97, untouched facts intact 0.98
  after  60 edits: earlier edits holding 0.90, untouched facts intact 0.95
  after  80 edits: earlier edits holding 0.85, untouched facts intact 1.00


## Questions

1. Why can an edit be perfect on its target and still be a bad edit? Use the numbers from Step 1.
2. The dependency in Step 2 was invisible to the edit. Where would such dependencies come from in a real language model, and how would you test for them before shipping an edited model?
3. A bank wants to correct one outdated product fact in its assistant. Based on Step 3, what would you tell it about editing versus retrieval of the current fact at answer time?